**UPLOADING DATASET**

In [1]:
from google.colab import files
uploaded = files.upload()

Saving netflix_titles.csv to netflix_titles.csv


**DATA** **CLEANING:**

In [2]:
import pandas as pd
df = pd.read_csv("netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


**RAW DATA:**

In [3]:
print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape:
(8807, 12)

Column Names:
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']

Data Types:
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

Missing Values:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

Duplicate Rows:
0


In [4]:
clean_df = df.copy()

In [5]:
clean_df["director"] = clean_df["director"].fillna("Unknown")
clean_df["cast"] = clean_df["cast"].fillna("Unknown")
clean_df["country"] = clean_df["country"].fillna("Unknown")

In [6]:
clean_df = clean_df.dropna(subset=["date_added", "rating", "duration"])

print("Shape after cleaning:", clean_df.shape
print("\nRemaining Missing Values:")
print(clean_df.isnull().sum())

Shape after cleaning: (8790, 12)

Remaining Missing Values:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [9]:
clean_df["date_added"] = clean_df["date_added"].str.strip()

clean_df["date_added"] = pd.to_datetime(
    clean_df["date_added"],
    format="%B %d, %Y"
)
print(clean_df["date_added"].dtype)
clean_df[["title", "date_added"]].head()

datetime64[ns]


,title,date_added
0,Dick Johnson Is Dead,2021-09-25
1,Blood & Water,2021-09-24
2,Ganglands,2021-09-24
3,Jailbirds New Orleans,2021-09-24
4,Kota Factory,2021-09-24


In [10]:
clean_df["year_added"] = clean_df["date_added"].dt.year
clean_df["month_added"] = clean_df["date_added"].dt.month
clean_df[["title", "date_added", "year_added", "month_added"]].head()

,title,date_added,year_added,month_added
0,Dick Johnson Is Dead,2021-09-25,2021,9
1,Blood & Water,2021-09-24,2021,9
2,Ganglands,2021-09-24,2021,9
3,Jailbirds New Orleans,2021-09-24,2021,9
4,Kota Factory,2021-09-24,2021,9


In [11]:
clean_df["content_age_when_added"] = (
    clean_df["year_added"] - clean_df["release_year"]
)

clean_df[
    ["title", "release_year", "year_added", "content_age_when_added"]
].head()

,title,release_year,year_added,content_age_when_added
0,Dick Johnson Is Dead,2020,2021,1
1,Blood & Water,2021,2021,0
2,Ganglands,2021,2021,0
3,Jailbirds New Orleans,2021,2021,0
4,Kota Factory,2021,2021,0


In [12]:
print("Negative values:")
print((clean_df["content_age_when_added"] < 0).sum())

print("\nMinimum content age:")
print(clean_df["content_age_when_added"].min())

Negative values:
14

Minimum content age:
-3


In [13]:
negative_age = clean_df[
    clean_df["content_age_when_added"] < 0
][["title", "type", "date_added", "year_added", "release_year", "content_age_when_added"]]

negative_age

,title,type,date_added,year_added,release_year,content_age_when_added
1551,Hilda,TV Show,2020-12-14,2020,2021,-1
1696,Polly Pocket,TV Show,2020-11-15,2020,2021,-1
2920,Love Is Blind,TV Show,2020-02-13,2020,2021,-1
3168,Fuller House,TV Show,2019-12-06,2019,2020,-1
3287,Maradona in Mexico,TV Show,2019-11-13,2019,2020,-1
3369,BoJack Horseman,TV Show,2019-10-25,2019,2020,-1
3433,The Hook Up Plan,TV Show,2019-10-11,2019,2020,-1
4844,Unbreakable Kimmy Schmidt,TV Show,2018-05-30,2018,2019,-1
4845,Arrested Development,TV Show,2018-05-29,2018,2019,-1
5394,Hans Teeuwen: Real Rancour,Movie,2017-07-01,2017,2018,-1


In [14]:
clean_df["content_age_when_added"] = clean_df["content_age_when_added"].clip(lower=0)

print("Negative values after correction:")
print((clean_df["content_age_when_added"] < 0).sum())

print("\nMinimum content age:")
print(clean_df["content_age_when_added"].min())

Negative values after correction:
0

Minimum content age:
0


In [15]:
clean_df["duration_value"] = clean_df["duration"].str.extract(r"(\d+)").astype(int)

clean_df["duration_unit"] = clean_df["duration"].str.extract(r"([A-Za-z]+)")

clean_df[
    ["title", "type", "duration", "duration_value", "duration_unit"]
].head()

,title,type,duration,duration_value,duration_unit
0,Dick Johnson Is Dead,Movie,90 min,90,min
1,Blood & Water,TV Show,2 Seasons,2,Seasons
2,Ganglands,TV Show,1 Season,1,Season
3,Jailbirds New Orleans,TV Show,1 Season,1,Season
4,Kota Factory,TV Show,2 Seasons,2,Seasons


**CLEANED DATA:**

In [16]:
print("Final Dataset Shape:", clean_df.shape)

print("\nMissing Values:")
print(clean_df.isnull().sum())

print("\nDuplicate Rows:")
print(clean_df.duplicated().sum())

print("\nData Types:")
print(clean_df.dtypes)

Final Dataset Shape: (8790, 17)

Missing Values:
show_id                   0
type                      0
title                     0
director                  0
cast                      0
country                   0
date_added                0
release_year              0
rating                    0
duration                  0
listed_in                 0
description               0
year_added                0
month_added               0
content_age_when_added    0
duration_value            0
duration_unit             0
dtype: int64

Duplicate Rows:
0

Data Types:
show_id                           object
type                              object
title                             object
director                          object
cast                              object
country                           object
date_added                datetime64[ns]
release_year                       int64
rating                            object
duration                          object
listed_in           

**EDA:**

In [19]:
#1 Analyze the Content Mix

content_type_counts = clean_df["type"].value_counts()
print(content_type_counts)
print("\nPercentage Distribution:")
print((clean_df["type"].value_counts(normalize=True) * 100).round(2))


type
Movie      6126
TV Show    2664
Name: count, dtype: int64

Percentage Distribution:
type
Movie      69.69
TV Show    30.31
Name: proportion, dtype: float64


In [20]:
#2 Analyze Netflix Content Growth Over Time

content_by_year = clean_df["year_added"].value_counts().sort_index()
print(content_by_year)

year_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     426
2017    1185
2018    1648
2019    2016
2020    1879
2021    1498
Name: count, dtype: int64


In [21]:
#3 Analyze the Top Countries

country_counts = (
    clean_df["country"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)
print(country_counts.head(10))

country
United States     3681
India             1046
Unknown            829
United Kingdom     805
Canada             445
France             393
Japan              316
Spain              232
South Korea        231
Germany            226
Name: count, dtype: int64


In [22]:
#4 Analyze Top Genres

genre_counts = (
    clean_df["listed_in"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print(genre_counts.head(10))

listed_in
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [23]:
#5 Analyze Content Ratings

rating_counts = clean_df["rating"].value_counts()
print(rating_counts)

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [24]:
#6 Analyze Movie Duration

movie_duration = clean_df[
    clean_df["type"] == "Movie"
]["duration_value"]

print("Average Movie Duration:", round(movie_duration.mean(), 2), "minutes")

print("Median Movie Duration:", movie_duration.median(), "minutes")

print("Minimum Duration:", movie_duration.min(), "minutes")

print("Maximum Duration:", movie_duration.max(), "minutes")

Average Movie Duration: 99.58 minutes
Median Movie Duration: 98.0 minutes
Minimum Duration: 3 minutes
Maximum Duration: 312 minutes


In [25]:
#7 Analyze Content Age When Added

print("Average Content Age:", round(clean_df["content_age_when_added"].mean(), 2), "years")
print("Median Content Age:", clean_df["content_age_when_added"].median(), "years")
print("\nContent Age Summary:")
print(clean_df["content_age_when_added"].describe())

Average Content Age: 4.69 years
Median Content Age: 1.0 years

Content Age Summary:
count    8790.000000
mean        4.692378
std         8.788835
min         0.000000
25%         0.000000
50%         1.000000
75%         5.000000
max        93.000000
Name: content_age_when_added, dtype: float64


**EXPORTING CLEANED DATASET**

In [29]:
clean_df.to_csv("netflix_cleaned.csv", index=False)

In [30]:
from google.colab import files
files.download("netflix_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>